# 03 · Part B — attribution + steering (needs GPU ≥24 GB)\nLoads Qwen2.5-Omni-7B Thinker (→ `$CAGA_BASE/models/hf`) and runs per-item grad×act attribution + activation steering, sweeping 4 modes × 6 K × 5 α. Resumable per (dataset, mode, K, α, item).

In [1]:
# add project root to sys.path (works whether cwd is project root or notebooks/)
import sys, os
for cand in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(cand, "caga")):
        if cand not in sys.path: sys.path.insert(0, cand)
        break

### Load model (once)

In [ ]:
from caga import partb
partb.load_model()

: 

In [3]:
import time

it = partb.get_items('mmar')[0]; ut = partb.build_user_text(it)
wav, contrast, _ = partb.load_audio_and_pair(it['audio_abs_path'])
t=time.time()
for _ in range(3): attr = partb.compute_scope_attribution(wav, contrast, ut, partb.M_GRADXACT)
t_attr=(time.time()-t)/3
gi = partb.prepare_gen_inputs(wav, ut)
specs = partb.build_specs(attr,'text',1000,partb.V30_NORM)
r = partb.PerItemRunner(partb.processor, partb.model, specs, attr['T'])
t=time.time()
for _ in range(3): r.gen_batch_from_inputs(gi, partb.V30_ALPHAS)
t_gen1=(time.time()-t)/3
t_gen_all = t_gen1 * len(partb.V30_K_LIST) * len(partb.V30_MODES)
print(f"attribution: {t_attr:.2f}s | generation (all configs): {t_gen_all:.2f}s | attribution share: {t_attr/(t_attr+t_gen_all)*100:.0f}%")

attribution: 2.26s | generation (all configs): 12.41s | attribution share: 15%


In [5]:
from caga import partb
partb.load_model()
partb.selftest_fast_attr('mmau_mini')   # should print MATCH
partb.selftest_alpha_batch('mmau_mini') # should print MATCH (the earlier batch check)

  layer 20: delta values differ
  layer 21: delta values differ
  layer 22: delta values differ
  layer 25: delta values differ
  layer 26: delta values differ
  layer 27: delta values differ
FAST_ATTR selftest: MISMATCH ✗ (set FAST_ATTR=False)
RESULT: MATCH ✓ (batched is safe)
  a=-1.0  seq='A'                  bat='A'                  ok
  a=-0.3  seq='a'                  bat='a'                  ok
  a=+0.0  seq='a'                  bat='a'                  ok
  a=+0.3  seq='a'                  bat='a'                  ok
  a=+1.0  seq='a'                  bat='a'                  ok


True

### Sanity-check

In [ ]:
from caga import partb
partb.load_model()
partb.selftest_alpha_batch('mmau_mini', mode='text', K=1000)

RESULT: MATCH ✓ (batched is safe)
  a=-1.0  seq='A'                  bat='A'                  ok
  a=-0.3  seq='a'                  bat='a'                  ok
  a=+0.0  seq='a'                  bat='a'                  ok
  a=+0.3  seq='a'                  bat='a'                  ok
  a=+1.0  seq='a'                  bat='a'                  ok


True

In [ ]:
partb.run_v30_2(['mmau_mini'])
partb.eval_v30_2(['mmau_mini'])


=== mmau_mini: 1000 items | modes=['final', 'audio', 'text', 'audio_text'] | K=[500, 1000, 5000, 10000, 20000, 50000] | 24 configs x 5 alphas | 2 fwd-bwd/item (gradxact) ===


  [sanity] id=mmau_mini/00084 grad-L2 audio/pos=0.00558 text/pos=0.04346 final=0.242 T=339
    [mmau_mini] 10 items (149s)
    [mmau_mini] 20 items (347s)
    [mmau_mini] 30 items (492s)
    [mmau_mini] 40 items (604s)
    [mmau_mini] 50 items (716s)
    [mmau_mini] 60 items (940s)
    [mmau_mini] 70 items (1180s)
    [mmau_mini] 80 items (1335s)
    [mmau_mini] 90 items (1448s)
    [mmau_mini] 100 items (1563s)
    [mmau_mini] 110 items (1672s)
    [mmau_mini] 120 items (1773s)
    [mmau_mini] 130 items (1868s)
    [mmau_mini] 140 items (1968s)
    [mmau_mini] 150 items (2089s)
    [mmau_mini] 160 items (2190s)
    [mmau_mini] 170 items (2317s)
    [mmau_mini] 180 items (2430s)
    [mmau_mini] 190 items (2543s)
    [mmau_mini] 200 items (2652s)
    [mmau_mini] 210 items (2752s)
    [mmau_mini] 220 items (2853s)
    [mmau_mini] 230 items (3034s)
    [mmau_mini] 240 items (3237s)
    [mmau_mini] 250 items (3477s)
    [mmau_mini] 260 items (3807s)
    [mmau_mini] 270 items (4057s)
    [m

In [4]:
partb.eval_v30_2(['mmau_mini'])


=== mmau_mini eval (norm=none, n=1000) ===
mode              K       a=m1     a=m0p3        a=0      a=0p3        a=1   best(a)        Δvs0    status
----------------------------------------------------------------------------------------------------------
final           500     0.6990     0.6990     0.6980     0.6950     0.6910    0.6990@-1.0   +0.0010   OK
final          1000     0.6980     0.6950     0.6980     0.6970     0.6870    0.6980@-1.0   +0.0000   OK


final          5000     0.6940     0.6970     0.6980     0.6960     0.6940    0.6970@-0.3   -0.0010   OK
final         10000     0.6960     0.6980     0.6980     0.6960     0.6930    0.6980@-0.3   +0.0000   OK
final         20000     0.6990     0.6960     0.6980     0.6950     0.6980    0.6990@-1.0   +0.0010   OK
final         50000     0.6950     0.6960     0.6980     0.6950     0.6930    0.6960@-0.3   -0.0020   OK

audio           500     0.6860     0.6950     0.6980     0.6930     0.6950    0.6950@-0.3   -0.0030   OK
audio          1000     0.6800     0.6900     0.6980     0.6940     0.6950    0.6950@+1.0   -0.0030   OK
audio          5000     0.6690     0.6880     0.6980     0.6920     0.6920    0.6920@+0.3   -0.0060   OK
audio         10000     0.6630     0.6860     0.6980     0.6890     0.6900    0.6900@+1.0   -0.0080   OK
audio         20000     0.6530     0.6880     0.6980     0.6960     0.7000    0.7000@+1.0   +0.0020   OK
audio         50000     0.6250     0.6880     0.6980  

### Then the rest (6 benchmarks). AIR-Bench can be added once `02` has populated it.

In [ ]:
#partb.run_v30_2(['mmau_mini','mmar','mmsu','ravdess','iemocap','meld'])
#partb.eval_v30_2(['mmau_mini','mmar','mmsu','ravdess','iemocap','meld'])

In [4]:
partb.selftest_fast_attr('mmau_mini', mode='text', K=1000, n_items=3)

=== FAST_ATTR GPU-vs-CPU diff over 3 items (mode=text, K=1000) ===
  neuron-set differences (selected neurons that changed): 2
  delta max ABSOLUTE diff : 4.219e+00
  delta max RELATIVE diff : 1.812e+08
  PREDICTION differences  : 0/15 letters
  => a few neurons differ at the top-K boundary but answers are identical; effect on accuracy ~0.


{'neuron_diff': 2,
 'max_abs': 4.21875,
 'max_rel': 181250000.0,
 'pred_diffs': 0,
 'pred_total': 15}

In [6]:
partb.run_v30_2(['mmar'])


=== mmar: 1000 items | modes=['final', 'audio', 'text', 'audio_text'] | K=[500, 1000, 5000, 10000, 20000, 50000] | 24 configs x 5 alphas | 2 fwd-bwd/item (gradxact) ===


  [sanity] id=mmar/BV1Bu4y1H7sW_00-01-56_00-02-14 grad-L2 audio/pos=0.00300 text/pos=0.07421 final=0.412 T=538
    [mmar] 10 items (166s)
    [mmar] 20 items (357s)
    [mmar] 30 items (529s)


KeyboardInterrupt: 

In [7]:
import torch
from torch.profiler import profile, ProfilerActivity
from caga import partb

# profile a single item
it = partb.get_items('mmau_mini')[0]
ut = partb.build_user_text(it)
wav, contrast, _ = partb.load_audio_and_pair(it['audio_abs_path'])

def one_item():
    attr = partb.compute_scope_attribution(wav, contrast, ut, partb.M_GRADXACT)
    gi = partb.prepare_gen_inputs(wav, ut)
    for K in [1000, 10000]:                    # smaller, enough to see the proportions
        for mode in ['text', 'audio']:
            specs = partb.build_specs(attr, mode, K, partb.V30_NORM)
            r = partb.PerItemRunner(partb.processor, partb.model, specs, attr['T'])
            r.gen_batch_from_inputs(gi, partb.V30_ALPHAS)

one_item()  # warmup

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
             record_shapes=False) as prof:
    one_item()

# sort by total CPU time, show the top 20
print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        cudaMemcpyAsync        12.62%     783.100ms        12.62%     783.100ms     400.153us       0.000us         0.00%       0.000us       0.000us          1957  
                                       cudaLaunchKernel         8.25%     512.202ms         8.25%     512.202ms      18.928us       0.000us         0.00%       0.000us       0.000us         27060  
         

In [3]:
import torch
from torch.profiler import profile, ProfilerActivity
from caga import partb

# profile a single item
it = partb.get_items('mmau_mini')[0]
ut = partb.build_user_text(it)
wav, contrast, _ = partb.load_audio_and_pair(it['audio_abs_path'])

def one_item():
    attr = partb.compute_scope_attribution(wav, contrast, ut, partb.M_GRADXACT)
    gi = partb.prepare_gen_inputs(wav, ut)
    for K in [1000, 10000]:                    # smaller, enough to see the proportions
        for mode in ['text', 'audio']:
            specs = partb.build_specs(attr, mode, K, partb.V30_NORM)
            r = partb.PerItemRunner(partb.processor, partb.model, specs, attr['T'])
            r.gen_batch_from_inputs(gi, partb.V30_ALPHAS)

one_item()  # warmup

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
             record_shapes=False) as prof:
    one_item()

# sort by total CPU time, show the top 20
print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                       cudaLaunchKernel         8.46%     294.610ms         8.46%     294.610ms      10.674us       0.000us         0.00%       0.000us       0.000us         27600  
                                  cudaStreamSynchronize         6.07%     211.575ms         6.07%     211.575ms     140.023us       0.000us         0.00%       0.000us       0.000us          1511  
         

In [4]:
partb.eval_v30_2(['meld'])


=== meld eval (norm=none, n=3718) ===
mode              K       a=m1     a=m0p3        a=0      a=0p3        a=1   best(a)        Δvs0    status
----------------------------------------------------------------------------------------------------------
final           500     0.4890     0.4966     0.5007     0.5048     0.5007              -         -   PARTIAL 1456/3718
final          1000     0.4945     0.5000     0.5007     0.5041     0.5021              -         -   PARTIAL 1456/3718


final          5000     0.4890     0.5034     0.5007     0.5014     0.4986              -         -   PARTIAL 1456/3718
final         10000     0.4897     0.5055     0.5007     0.5027     0.4918              -         -   PARTIAL 1456/3718
final         20000     0.4876     0.5062     0.5007     0.4973     0.4959              -         -   PARTIAL 1456/3718
final         50000     0.4870     0.5062     0.5007     0.4979     0.4973              -         -   PARTIAL 1456/3718

audio           500     0.4402     0.5021     0.5007     0.4863     0.4698              -         -   PARTIAL 1456/3718
audio          1000     0.4203     0.5034     0.5007     0.4897     0.4753              -         -   PARTIAL 1456/3718
audio          5000     0.3915     0.5021     0.5007     0.4890     0.4739              -         -   PARTIAL 1456/3718
audio         10000     0.3736     0.5041     0.5007     0.4911     0.4746              -         -   PARTIAL 1456/3718
audio         20000     0.3668     0.50

In [2]:
from caga import partb
partb.eval_v30_2(['mmau_mini','mmar','mmsu','ravdess','iemocap','meld'])

$CAGA_BASE/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


config ready | contrast = gaussian_pure | alphas = [-1.0, -0.3, 0.0, 0.3, 1.0] | RUNS_BASE = $CAGA_BASE/caga_v30_2_runs
forward helpers ready (gaussian 'pure' contrast)
v30 methods ready: compute_scope_attribution (gradxact) / build_specs (modes) / PerItemRunner

=== mmau_mini eval (norm=none, n=1000) ===
mode              K       a=m1     a=m0p3        a=0      a=0p3        a=1   best(a)        Δvs0    status
----------------------------------------------------------------------------------------------------------
final           500     0.6990     0.6990     0.6980     0.6950     0.6910    0.6990@-1.0   +0.0010   OK
final          1000     0.6980     0.6950     0.6980     0.6970     0.6870    0.6980@-1.0   +0.0000   OK
final          5000     0.6940     0.6970     0.6980     0.6960     0.6940    0.6970@-0.3   -0.0010   OK
final         10000     0.6960     0.6980     0.6980     0.6960     0.6930    0.6980@-0.3   +0.0000   OK
final         20000     0.6990     0.6960     0.6980     0.